In [72]:
import pandas as pd
import numpy as np
import pyshark
import asyncio
import ssl

DATA_FOLDER = "data"
PCAP_FILE = f"{DATA_FOLDER}/trace.pcap"
TOP_URLS_FILE = f"{DATA_FOLDER}/top_urls.csv"

OUTPUT_FOLDER = f"output"
OUTPUT_FILE = f"{OUTPUT_FOLDER}/traces.csv"

In [5]:
top_urls = pd.read_csv(TOP_URLS_FILE)
https_urls = top_urls[top_urls["url"].str.startswith("https://")]
https_urls

,url
2,https://18comic.vip
3,https://7games.bet.br
4,https://8moviesda.net
5,https://9animetv.to
6,https://9moviesda.com
...,...
995,https://zbporn.tv
996,https://zeenews.india.com
997,https://zh.m.wikipedia.org
998,https://zonatmo.com


In [12]:
context = ssl.create_default_context()
i = 1


async def good_url(url):
    domain = url.removeprefix("https://")
    try:
        _, writer = await asyncio.wait_for(asyncio.open_connection(domain, 443), timeout=25)
        await writer.start_tls(
            context, server_hostname=domain, ssl_handshake_timeout=5, ssl_shutdown_timeout=5
        )

        global i
        if i % 50 == 0:
            print(f"{i} good URLs so far...")
        i += 1

        return True
    except Exception as e:
        if str(e):
            print(f"Error connecting {domain}: {e}")
        else:
            print(f"Error connecting {domain}: {repr(e)}")
        return False


good_urls_indices = await asyncio.gather(*[good_url(url) for url in https_urls["url"].tolist()])
good_urls = https_urls[good_urls_indices]

Error connecting bollyflix.tw: [Errno -2] Name or service not known
Error connecting cricbet99.club: [Errno -5] No address associated with hostname
50 good URLs so far...
100 good URLs so far...
150 good URLs so far...
200 good URLs so far...
Error connecting hrms.indianrail.gov.in: [Errno 111] Connect call failed ('203.176.112.88', 443)
250 good URLs so far...
Error connecting moviesda.it.com: [Errno -5] No address associated with hostname
300 good URLs so far...
Error connecting new2.filesdl.site: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)
350 good URLs so far...
400 good URLs so far...
Error connecting streamable.cloud: [Errno -2] Name or service not known
Error connecting service.smt.docomo.ne.jp: [SSL: UNSAFE_LEGACY_RENEGOTIATION_DISABLED] unsafe legacy renegotiation disabled (_ssl.c:1028)
Error connecting smt.docomo.ne.jp: [SSL: UNSAFE_LEGACY_RENEGOTIATION_DISABLED] unsafe legacy renegotiation disabled (_ssl.c:

In [57]:
data = []


def process_packet(packet):
    data.append(
        {
            "timestamp": packet.sniff_time,
            "domain": packet.tls.handshake_extensions_server_name,
        }
    )


# Only keep TLS Client Hello packets
cap = pyshark.FileCapture(PCAP_FILE, display_filter="tls.handshake.type == 1")
await cap.packets_from_tshark(process_packet)

In [58]:
data = pd.DataFrame(data)
# Express timestamps as second offset since the first packet
initial_time = data["timestamp"].min()
data["timestamp"] = (data["timestamp"] - initial_time).dt.total_seconds()
data

,timestamp,domain
0,0.000000,www.yahoo.com
1,0.283730,init.push.apple.com
2,0.603494,ncp-gw-frontpage.media.yahoo.com
3,0.603663,nexus-gateway-prod.media.yahoo.com
4,0.604100,s.yimg.com
...,...,...
310,2069.563360,o64374.ingest.sentry.io
311,2069.649936,gateway.discord.gg
312,2071.098055,dealer.spotify.com
313,2072.683935,cdn.discordapp.com


In [66]:
mapping = {}

good_urls = good_urls.sample(frac=1).reset_index(drop=True)
index = 0
for domain in data["domain"].unique():
    mapping[domain] = good_urls["url"].iloc[index]
    index += 1
    if index >= len(good_urls):
        index = 0

data_mapped = data.copy()
data_mapped["url"] = data_mapped["domain"].map(mapping)
data_mapped = data_mapped.drop(columns=["domain"])
data_mapped

,timestamp,url
0,0.000000,https://www.goodreturns.in
1,0.283730,https://search.naver.com
2,0.603494,https://www2.hm.com
3,0.603663,https://www.amazon.in
4,0.604100,https://bbs.animanch.com
...,...,...
310,2069.563360,https://skipthegames.com
311,2069.649936,https://m.dcinside.com
312,2071.098055,https://www.metropoles.com
313,2072.683935,https://www.arbada.com


In [67]:
BUCKET_SIZE = 30 * 60  # 30 mins

data_grouped = data_mapped
data_grouped["group"] = (data_grouped["timestamp"] // BUCKET_SIZE).astype(int)
data_grouped.iloc[np.r_[0:5, -5:0]]

,timestamp,url,group
0,0.000000,https://www.goodreturns.in,0
1,0.283730,https://search.naver.com,0
2,0.603494,https://www2.hm.com,0
3,0.603663,https://www.amazon.in,0
4,0.604100,https://bbs.animanch.com,0
310,2069.563360,https://skipthegames.com,1
311,2069.649936,https://m.dcinside.com,1
312,2071.098055,https://www.metropoles.com,1
313,2072.683935,https://www.arbada.com,1
314,2077.242512,https://vailonxx.vip,1


In [73]:
BUCKETS_TO_KEEP = 1

# we will keep the most populous buckets
group_sizes = data_grouped.groupby("group").size()
data_pruned = data_grouped[
    data_grouped["group"].isin(group_sizes.nlargest(BUCKETS_TO_KEEP).index.tolist())
]
print(f"Largest bucket has {data_pruned.groupby('group').size().max()} packets")
print(f"Smallest bucket has {data_pruned.groupby('group').size().min()} packets")
data_pruned

Largest bucket has 268 packets
Smallest bucket has 268 packets


,timestamp,url,group
0,0.000000,https://www.goodreturns.in,0
1,0.283730,https://search.naver.com,0
2,0.603494,https://www2.hm.com,0
3,0.603663,https://www.amazon.in,0
4,0.604100,https://bbs.animanch.com,0
...,...,...,...
263,1715.710387,https://classroom.google.com,0
264,1716.124739,https://www.rojgarresult.com,0
265,1716.198402,https://gemini.google.com,0
266,1717.106430,https://people.com,0


In [74]:
data_pruned.rename(columns={"group": "trace"}).to_csv(OUTPUT_FILE, index=False)